# Optimization of food delivery by drones in Sierre's district

## Imports

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Tuple, Dict
from rich.console import Console
from IPython.display import clear_output
import json

from utils.data_loader import data_loader
from utils.utils import get_demand_mtx, cost_fct
from utils.solver import solve_cvrp
from utils.visualization import render_terminal_dashboard, plot_gantt_chart, plot_demand_curves

In [3]:
def full_loop(distances_df : pd.DataFrame, altitudes_df : pd.DataFrame, commune_info_df : pd.DataFrame, hourly_weekly_demand_df : pd.DataFrame,
              nb_drones: int = 30, demand_threshold: float = 0.0, maximum_operating_hours=78, 
              penalization: bool = False, penalty_per_unserved_order: float = 10.0) -> Tuple[float, List[Dict]]:
    """
    Executes the full simulation loop for the drone delivery system, iterating through each day of the week and each 15-minute chunk of time to
    optimize drone routes based on demand and operational constraints.
    
    :param distances_df: DataFrame containing distances between communes.
    :type distances_df: pd.DataFrame
    :param altitudes_df: DataFrame containing altitude information for communes.
    :type altitudes_df: pd.DataFrame
    :param commune_info_df: DataFrame containing information about communes, including demand.
    :type commune_info_df: pd.DataFrame
    :param hourly_weekly_demand_df: DataFrame containing hourly share of daily demand for each day of the week.
    :type hourly_weekly_demand_df: pd.DataFrame
    :param nb_drones: Number of drones available for delivery.
    :type nb_drones: int, default=30
    :param demand_threshold: Minimum demand threshold to consider for delivery.
    :type demand_threshold: float, default=0.0
    :param maximum_operating_hours: Maximum operating hours for drones per week.
    :type maximum_operating_hours: int, default=78
    :param penalization: Whether to apply penalization for unserved orders.
    :type penalization: bool, default=False
    :param penalty_per_unserved_order: Penalty cost for each unserved order if penalization is enabled.
    :type penalty_per_unserved_order: float, default=10.0
    :return: A tuple containing the final cost of the simulation and a log of drone activities for visualization.
    :rtype: Tuple[float, List[Dict]]
    """
    remaining_flying_time_per_drones = [0.0 for _ in range(nb_drones)]
    max_load = 12 
    # battery_capacity = 2131 
    battery_capacity = 3000 
    weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    
    total_energy_consumed = 0.0
    total_penalty_cost_chf = 0.0
    commune_names = commune_info_df.index.tolist()
    total_successful_deliveries = 0
    total_missed_deliveries = 0
    
    # Initialize Presentation Tools
    console = Console()
    drone_activity_log = []
    simulation_start = datetime(2026, 6, 1, 0, 0) # Use an arbitrary Monday for clean Plotly dates

    # Initialize the master history list
    simulation_history = []

    for day_idx, d in enumerate(weekdays):
        for h in range(10, 25): 
            for chunk in range(4): 
                
                # Time calculation for Plotly and Display
                current_time = simulation_start + timedelta(days=day_idx, hours=h-10, minutes=chunk*15)
                
                # 1. Update active drones and decrement remaining times by 15 mins BEFORE the chunk
                remaining_flying_time_per_drones = [max(0.0, t - 15.0) for t in remaining_flying_time_per_drones]
                available_drones = [True if t == 0.0 else False for t in remaining_flying_time_per_drones]
                
                # 2. Fetch Demand
                round_demand = get_demand_mtx(h, d, demand_threshold, hourly_weekly_demand_df, commune_info_df) 
                
                # 3. Solve 15-minute chunk
                routes, distance_per_drone, round_energy, remaining_flying_time_per_drones, route_times, dropped_orders, used_drone_ids = solve_cvrp(
                    available_vehicles=available_drones, 
                    round_demand=round_demand, 
                    remaining_flying_time_per_drones=remaining_flying_time_per_drones, 
                    max_load=max_load, 
                    battery_capacity=battery_capacity, 
                    penalization=penalization, 
                    penalty_per_unserved_order=penalty_per_unserved_order,
                    commune_names=commune_names,
                    distances_df=distances_df,
                    altitudes_df=altitudes_df
                )
                                
                # 4. Process Data & Log Flights for Gantt Chart
                round_energy_wh = sum(round_energy)
                total_energy_consumed += round_energy_wh

                total_requested_this_round = int(np.sum(round_demand))
                served_this_round = total_requested_this_round - dropped_orders
                
                total_successful_deliveries += served_this_round
                total_missed_deliveries += dropped_orders

                total_penalty_cost_chf = 0.0
                if penalization and dropped_orders > 0:
                    total_penalty_cost_chf += (dropped_orders * penalty_per_unserved_order)
                
                active_indices = [i for i, is_avail in enumerate(available_drones) if is_avail]
                
                for idx, r_time in enumerate(route_times):
                    # Use used_drone_ids instead of active_indices!
                    drone_id = f"D{used_drone_ids[idx]:02d}"
                    start_dt = current_time
                    end_dt = current_time + timedelta(minutes=r_time)
                    drone_activity_log.append({
                        "Drone": drone_id,
                        "Start": start_dt,
                        "Finish": end_dt,
                        "Status": "Flying",
                        "Path": " -> ".join(routes[idx])
                    })
                
                round_snapshot = {
                    "timestamp": current_time.isoformat(), # Convert datetime to string for JSON
                    "day": d,
                    "hour_chunk": f"{h:02d}:{(chunk*15):02d}",
                    "financials": {
                        "round_energy_cost_chf": round_energy_wh * 0.00027,
                        "cumulative_energy_cost_chf": total_energy_consumed * 0.00027,
                        "penalties_incurred_chf": dropped_orders * penalty_per_unserved_order if penalization else 0.0
                    },
                    "metrics": {
                        "successful_deliveries": served_this_round,
                        "missed_deliveries": dropped_orders,
                        "cumulative_successful": total_successful_deliveries,
                        "cumulative_missed": total_missed_deliveries
                    },
                    "fleet_status": {
                        # Store as a list of dictionaries for easy parsing
                        "drones": [{"id": i, "minutes_until_ready": t} for i, t in enumerate(remaining_flying_time_per_drones)]
                    },
                    "active_routes": []
                }

                # Populate the active routes for this specific round
                if routes:
                    for idx, route in enumerate(routes):
                        round_snapshot["active_routes"].append({
                            "drone_id": used_drone_ids[idx],
                            "path": route, # e.g., ["warehouse", "CommuneA", "warehouse"]
                            "energy_used_wh": round_energy[idx],
                            "distance_km": distance_per_drone[idx]
                        })

                # Save the snapshot to our master timeline
                simulation_history.append(round_snapshot)

                # 5. Render live dashboard
                render_terminal_dashboard(current_time, round_energy_wh, total_energy_consumed, 
                                          routes, used_drone_ids, remaining_flying_time_per_drones, 
                                          dropped_orders, penalty_per_unserved_order, 
                                          total_successful_deliveries, total_missed_deliveries, console)

    # Calculate Total System Cost
    final_cost = cost_fct(weekly_operating_minutes=maximum_operating_hours * 60, 
                          total_energy_consumed=total_energy_consumed, 
                          n_drones=nb_drones)
    final_cost += total_penalty_cost_chf                
    # The console print here persists at the end of the simulation
    clear_output(wait=True)
    output_filename = "simulation_results.json"
    with open(output_filename, "w") as f:
        json.dump(simulation_history, f, indent=4)
        
    console.print(f"[bold green]💾 Simulation state saved successfully to {output_filename}[/bold green]")
        
    return final_cost, drone_activity_log

In [4]:
altitudes_df, distances_df, hourly_weekly_demand_df, commune_info_df = data_loader()
# plot_demand_curves(hourly_weekly_demand_df, commune_info_df)

In [5]:
final_cost, drone_log = full_loop(distances_df, altitudes_df, commune_info_df, hourly_weekly_demand_df)

print(final_cost)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🕰️ Time: Monday 12:00                💸 Round Cost: CHF 0.2859             💰 Total Cost: CHF 66.1096           │
│ 📦 Total Delivered: 1040             ❌ Total Missed: 0                    📊 Success Rate: 100.0%              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                        🚁 Active Routes this 15-Min Chunk                                         
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Drone ID           ┃ Flight Path                                                                                ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Drone 27           │ Warehouse ➔ Sierre ➔ Sierre ➔ Crans-Montana ➔ Warehouse                                    │
└────────────────────┴────────────────────────────────────────────────────────────────────────────────────────────┘

                                    ⏱️ Fleet Availability (Minutes until ready)                                    
┌──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ D00: READY   │ D01: READY   │ D02: READY  │ D03: READY  │ D04: READY  │ D05: READY  │ D06: READY  │ D07: READY  │
│ D08: READY   │ D09: READY   │ D10: READY  │ D11: READY  │ D12: READY  │ D13: READY  │ D14: READY  │ D15: READY  │
│ D16: READY   │ D17: READY   │ D18: READY  │ D19: READY  │ D20: READY  │ D21: READY  │ D22: 14.0m  │ D23: 21.0m  │
│ D24: 59.0m   │ D25: 6.0m    │ D26: READY  │ D27: 37.0m  │ D28: 5.0m   │ D29: 18.0m  │             │             │
└──────────────┴──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘

Solving next chunk

: 

: 

In [ ]:
plot_gantt_chart(drone_log)

In [2]:
def simulation(distances_df: pd.DataFrame, altitudes_df: pd.DataFrame, commune_info_df: pd.DataFrame, hourly_weekly_demand_df: pd.DataFrame,
              nb_drones: int = 30, scenario_id: int = 1, penalization: bool = False, penalty_per_unserved_order: float = 10.0, simulation_nb : str = "") -> Tuple[float, List[Dict]]:
    
    weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    
    scenarios = {
        1: {
            "name": "Full Operation (Extended Weekends)",
            "max_operating_hours": 100,
            "demand_threshold": 0.0,
            "schedule": {
                "Monday": list(range(10, 24)),    # 10:00 to 23:59
                "Tuesday": list(range(10, 24)),
                "Wednesday": list(range(10, 24)),
                "Thursday": list(range(10, 24)),
                "Friday": list(range(10, 25)),    # 10:00 to 00:59 (Includes hour 24)
                "Saturday": list(range(10, 25)),  # 10:00 to 00:59
                "Sunday": list(range(10, 24))
            }
        },
        2: {
            "name": "Peak Hours Only (Lunch & Dinner)",
            "max_operating_hours": 42,
            "demand_threshold": 0.0,
            # 12, 13, 14 (Lunch) and 18, 19, 20, 21, 22 (Dinner)
            "schedule": {day: [12, 13, 14, 18, 19, 20, 21, 22] for day in weekdays}
        }
    }

    if scenario_id not in scenarios:
        raise ValueError(f"Scenario {scenario_id} is not defined.")
        
    config = scenarios[scenario_id]
    active_schedule = config["schedule"]
    demand_threshold = config["demand_threshold"]
    maximum_operating_hours = config["max_operating_hours"]
    
    remaining_flying_time_per_drones = [0.0 for _ in range(nb_drones)]
    max_load = 12 
    battery_capacity = 4000 
    
    total_energy_consumed = 0.0
    total_penalty_cost_chf = 0.0
    total_successful_deliveries = 0
    total_missed_deliveries = 0
    
    commune_names = commune_info_df.index.tolist()
    console = Console()
    simulation_history = []
    drone_activity_log = []
    
    # Use Monday, June 1, 2026 as the anchor for clean datetime math
    simulation_start = datetime(2026, 6, 1, 0, 0) 
    previous_time = None # Used to track real elapsed time between jumps

    console.print(f"[bold green]🚀 Starting Scenario {scenario_id}: {config['name']}[/bold green]")

    for day_idx, d in enumerate(weekdays):
        active_hours = active_schedule[d] # Get the specific hours for this day
        
        for h in active_hours: 
            for chunk in range(4): 
                
                # 1. Calculate Exact Current Time
                current_time = simulation_start + timedelta(days=day_idx, hours=h, minutes=chunk*15)
                
                # 2. Dynamic Cooldown tracking (Fixes the gap between 14:00 and 18:00)
                if previous_time is not None:
                    elapsed_minutes = (current_time - previous_time).total_seconds() / 60.0
                else:
                    elapsed_minutes = 15.0 # Default for the very first chunk
                    
                previous_time = current_time
                
                # Apply the elapsed time to the drones
                remaining_flying_time_per_drones = [max(0.0, t - elapsed_minutes) for t in remaining_flying_time_per_drones]
                available_drones = [True if t == 0.0 else False for t in remaining_flying_time_per_drones]
                
                # 3. Fetch Demand
                round_demand = get_demand_mtx(h, d, demand_threshold, hourly_weekly_demand_df, commune_info_df) 
                
                # 4. Solve 15-minute chunk
                routes, distance_per_drone, round_energy, remaining_flying_time_per_drones, route_times, dropped_orders, used_drone_ids = solve_cvrp(
                    available_vehicles=available_drones, 
                    round_demand=round_demand, 
                    remaining_flying_time_per_drones=remaining_flying_time_per_drones, 
                    max_load=max_load, 
                    battery_capacity=battery_capacity, 
                    penalization=penalization, 
                    penalty_per_unserved_order=penalty_per_unserved_order,
                    commune_names=commune_names,
                    distances_df=distances_df,
                    altitudes_df=altitudes_df
                )
                
                # 5. Process Data & Financials
                round_energy_wh = sum(round_energy)
                total_energy_consumed += round_energy_wh
                
                total_requested_this_round = int(np.sum(round_demand))
                served_this_round = total_requested_this_round - dropped_orders
                
                total_successful_deliveries += served_this_round
                total_missed_deliveries += dropped_orders
                
                if penalization and dropped_orders > 0:
                    total_penalty_cost_chf += (dropped_orders * penalty_per_unserved_order)
                
                # 6. JSON Snapshot Generation
                round_snapshot = {
                    "timestamp": current_time.isoformat(),
                    "day": d,
                    "hour_chunk": f"{h:02d}:{(chunk*15):02d}",
                    "financials": {
                        "round_energy_cost_chf": round_energy_wh * 0.00027,
                        "cumulative_energy_cost_chf": total_energy_consumed * 0.00027,
                        "penalties_incurred_chf": (dropped_orders * penalty_per_unserved_order) if penalization else 0.0
                    },
                    "metrics": {
                        "successful_deliveries": served_this_round,
                        "missed_deliveries": dropped_orders,
                        "cumulative_successful": total_successful_deliveries,
                        "cumulative_missed": total_missed_deliveries
                    },
                    "fleet_status": {
                        "drones": [{"id": i, "minutes_until_ready": t} for i, t in enumerate(remaining_flying_time_per_drones)]
                    },
                    "active_routes": []
                }

                if routes:
                    for idx, route in enumerate(routes):
                        round_snapshot["active_routes"].append({
                            "drone_id": used_drone_ids[idx],
                            "path": route, 
                            "energy_used_wh": round_energy[idx],
                            "distance_km": distance_per_drone[idx]
                        })
                        
                        # Add to Plotly Gantt Log
                        drone_activity_log.append({
                            "Drone": f"D{used_drone_ids[idx]:02d}",
                            "Start": current_time,
                            "Finish": current_time + timedelta(minutes=route_times[idx]),
                            "Status": "Flying",
                            "Path": " -> ".join(route)
                        })

                simulation_history.append(round_snapshot)

                # 7. Render live dashboard
                render_terminal_dashboard(current_time, round_energy_wh, total_energy_consumed, 
                                          routes, used_drone_ids, remaining_flying_time_per_drones, 
                                          dropped_orders, penalty_per_unserved_order, 
                                          total_successful_deliveries, total_missed_deliveries, console)

    final_cost = cost_fct(weekly_operating_minutes=maximum_operating_hours * 60, 
                          total_energy_consumed=total_energy_consumed, 
                          n_drones=nb_drones)
    
    final_cost += total_penalty_cost_chf 
    
    output_filename = f"simulation_results_scenario_{scenario_id}_{simulation_nb}.json"
    with open(output_filename, "w") as f:
        json.dump(simulation_history, f, indent=4)
        
    clear_output(wait=True)
    console.print(f"\n[bold green]✅ Simulation Complete. Final Weekly Cost (Incl. Penalties): CHF {final_cost:,.2f}[/bold green]")
    console.print(f"[bold blue]💾 Simulation state saved successfully to {output_filename}[/bold blue]")
    
    return final_cost, total_successful_deliveries, drone_activity_log

In [4]:
altitudes_df, distances_df, hourly_weekly_demand_df, commune_info_df = data_loader()
final_cost, total_successful_deliveries, drone_activity_log = simulation(distances_df, altitudes_df, commune_info_df, hourly_weekly_demand_df, scenario_id=2, penalty_per_unserved_order=15.0, simulation_nb="test")

✅ Simulation Complete. Final Weekly Cost (Incl. Penalties): CHF 23,170.93

💾 Simulation state saved successfully to simulation_results_scenario_2_test.json

In [6]:
final_cost, total_successful_deliveries, final_cost/total_successful_deliveries

(np.float64(23170.927405573617), 5968, np.float64(3.8825280505317723))

In [9]:
plot_gantt_chart(drone_activity_log)

In [ ]:
altitudes_df, distances_df, hourly_weekly_demand_df, commune_info_df = data_loader()
costs, n_deliveries = [], []
for i in range(10):
    final_cost, total_successful_deliveries, drone_activity_log = simulation(distances_df, altitudes_df, commune_info_df, hourly_weekly_demand_df, simulation_nb=str(i))
    costs.append(final_cost)
    n_deliveries.append(total_successful_deliveries)
    print(f"Simulation {i} - Final Cost: CHF {final_cost:,.2f}, Total Successful Deliveries: {total_successful_deliveries}, average cost per delivery: CHF {(final_cost/total_successful_deliveries) if total_successful_deliveries > 0 else float('inf'):.2f}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🕰️ Time: Monday 11:30               💸 Round Cost: CHF 1.7637              💰 Total Cost: CHF 6.6054            │
│ 📦 Total Delivered: 87              ❌ Total Missed: 0                     📊 Success Rate: 100.0%              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                        🚁 Active Routes this 15-Min Chunk                                         
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Drone ID   ┃ Flight Path                                                                                        ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Drone 04   │ Warehouse ➔ Grône ➔ Saint-Léonard ➔ Lens ➔ Lens ➔ Crans-Montana ➔ Crans-Montana ➔ Crans-Montana ➔  │
│            │ Crans-Montana ➔ Crans-Montana ➔ Warehouse                                                          │
│ Drone 06   │ Warehouse ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Noble-Contrée  │
│            │ ➔ Noble-Contrée ➔ Warehouse                                                                        │
│ Drone 07   │ Warehouse ➔ Chalais ➔ Anniviers ➔ Warehouse                                                        │
└────────────┴────────────────────────────────────────────────────────────────────────────────────────────────────┘

                                    ⏱️ Fleet Availability (Minutes until ready)                                    
┌──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ D00: 35.0m   │ D01: 78.0m   │ D02: 78.0m  │ D03: 48.0m  │ D04: 73.0m  │ D05: READY  │ D06: 72.0m  │ D07: 50.0m  │
│ D08: READY   │ D09: READY   │ D10: READY  │ D11: READY  │ D12: READY  │ D13: READY  │ D14: READY  │ D15: READY  │
│ D16: READY   │ D17: READY   │ D18: READY  │ D19: READY  │ D20: READY  │ D21: READY  │ D22: READY  │ D23: READY  │
│ D24: READY   │ D25: READY   │ D26: READY  │ D27: 83.0m  │ D28: READY  │ D29: READY  │             │             │
└──────────────┴──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘

Solving next chunk

In [ ]:
for i in range(10):
    final_cost, total_successful_deliveries = costs[i], n_deliveries[i]
    print(f"Simulation {i} - Final Cost: CHF {final_cost:,.2f}, Total Successful Deliveries: {total_successful_deliveries}, average cost per delivery: CHF {(final_cost/total_successful_deliveries) if total_successful_deliveries > 0 else float('inf'):.2f}")